# 流式传输-概述

从智能体运行中流式实时更新

LangChain 实现了一个流式系统，用于实时更新。

流式传输对于提升基于大型语言模型（LLM）的应用程序响应性至关重要。通过逐步显示输出，甚至在完整响应准备就绪之前，流式传输显著提升了用户体验（UX），尤其是在处理LLM延迟问题时。

## 1. 概述
LangChain的流式传输系统允许将智能体运行的实时反馈呈现给应用。

LangChain 流式传输的可能性：
- 流代理进展——在每个智能体步骤后获取状态更新。
- 流式LLM tokens—— 在生成时流式语言模型tokens。
- 流式自定义更新—— 发送用户定义信号（例如，`"已获取 10/100 条记录"`）。
- 流式传输多种模式 — 从 `updates` (agent 进度)、`messages` (LLM token + 元数据) 或 `custom` (任意用户数据) 中选择。

## 2. 支持的流式传输模式
将以下一种或多种流式传输模式作为列表传递给`stream`或`astream`方法：
| 模式      | 描述                                                                 |
|:-----------|:----------------------------------------------------------------------|
| `updates`  | Streams 在每个智能体步骤后进行状态更新。如果同一步进行多次更新（例如运行多个节点），这些更新会分别流式传输。 |
| `messages` | 从调用 LLM 的任意图节点流出的元组 `(token, metadata)`。               |
| `custom`   | 通过流编写器从图节点内部流式传输自定义数据。                          |
## 3. Agent进度
要流式传输智能体进度，可以使用stream或astream方法 。每执行一个智能体步骤后都会发出一个事件。`stream_mode="updates"`

例如，如果有一个智能体只调用一次工具，应该会看到以下更新：
- LLM节点：AIMessage带有工具调用请求
- 工具节点：ToolMessage执行结果为
- LLM节点：最终AI响应

In [1]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

def get_weather(city: str) -> str:
    """Get weather for a given city."""

    return f"It's always sunny in {city}!"

model = ChatOllama(model="qwen3:0.6b")

agent = create_agent(
    model=model,
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="updates",
):
    for step, data in chunk.items():
        print(f"step: {step}")
        print(f"content: {data['messages'][-1].content_blocks}")

step: model
content: [{'type': 'tool_call', 'id': '77b08faf-d1f3-4065-b437-ecffff467b5b', 'name': 'get_weather', 'args': {'city': 'SF'}}]
step: tools
content: [{'type': 'text', 'text': "It's always sunny in SF!"}]
step: model
content: [{'type': 'text', 'text': "It's always sunny in SF! 🌤️  \nLet me know if you need more info!"}]


## 4. LLM Tokens
要流式传输 LLM 生成的 tokens，可以使用 `stream_mode="messages"`。可以在下面看到 agent 流式传输工具调用和最终响应的输出。

In [4]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

def get_weather(city: str) -> str:
    """Get weather for a given city."""

    return f"It's always sunny in {city}!"

agent = create_agent(
    model=model,
    tools=[get_weather],
)

for token, metadata in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="messages",
):
    if token.content_blocks:
        print(f"node: {metadata['langgraph_node']}")
        print(f"content: {token.content_blocks}")
        # print("\n")

node: model
content: [{'type': 'tool_call_chunk', 'id': '2ed39a80-a5ac-40d0-a587-c61047f0ef6e', 'name': 'get_weather', 'args': '{"city": "SF"}'}]
node: tools
content: [{'type': 'text', 'text': "It's always sunny in SF!"}]
node: model
content: [{'type': 'text', 'text': 'It'}]
node: model
content: [{'type': 'text', 'text': "'s"}]
node: model
content: [{'type': 'text', 'text': ' always'}]
node: model
content: [{'type': 'text', 'text': ' sunny'}]
node: model
content: [{'type': 'text', 'text': ' in'}]
node: model
content: [{'type': 'text', 'text': ' San'}]
node: model
content: [{'type': 'text', 'text': ' Francisco'}]
node: model
content: [{'type': 'text', 'text': '!'}]
node: model
content: [{'type': 'text', 'text': ' 🌞'}]
node: model
content: [{'type': 'text', 'text': ' Let'}]
node: model
content: [{'type': 'text', 'text': ' me'}]
node: model
content: [{'type': 'text', 'text': ' know'}]
node: model
content: [{'type': 'text', 'text': ' if'}]
node: model
content: [{'type': 'text', 'text': ' y

## 5. 自定义更新
要流式传输工具执行时的更新，可以使用 `get_stream_writer`。

In [5]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
from langchain_ollama import ChatOllama

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    # stream any arbitrary data
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"

model = ChatOllama(model="qwen3:0.6b")

agent = create_agent(
    model=model,
    tools=[get_weather],
)

for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode="custom"
):
    print(chunk)

Looking up data for city: San Francisco
Acquired data for city: San Francisco


## 6. 流式传输多种模式
可以通过将流模式作为列表传递来指定多种流模式：stream_mode=["updates", "custom"]

流式输出将是元组，其中 mode 是流模式的名称，chunk 是该模式流式传输的数据。

In [7]:
from langchain.agents import create_agent
from langgraph.config import get_stream_writer
from langchain_ollama import ChatOllama

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    writer = get_stream_writer()
    writer(f"Looking up data for city: {city}")
    writer(f"Acquired data for city: {city}")
    return f"It's always sunny in {city}!"

agent = create_agent(
    model=model,
    tools=[get_weather],
)

for stream_mode, chunk in agent.stream(
    {"messages": [{"role": "user", "content": "What is the weather in SF?"}]},
    stream_mode=["updates", "custom"]
):
    print(f"stream_mode: {stream_mode}")
    print(f"content: {chunk}")
    # print("\n")

stream_mode: updates
content: {'model': {'messages': [AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen3:0.6b', 'created_at': '2026-01-16T09:06:42.9775152Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2548656200, 'load_duration': 133110400, 'prompt_eval_count': 143, 'prompt_eval_duration': 43576400, 'eval_count': 102, 'eval_duration': 2343519100, 'logprobs': None, 'model_name': 'qwen3:0.6b', 'model_provider': 'ollama'}, id='lc_run--019bc60e-8aaa-7473-852a-50a63aa576f2-0', tool_calls=[{'name': 'get_weather', 'args': {'city': 'SF'}, 'id': 'fa6c4e23-d377-4fff-a4e3-da93561666af', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 143, 'output_tokens': 102, 'total_tokens': 245})]}}
stream_mode: custom
content: Looking up data for city: SF
stream_mode: custom
content: Acquired data for city: SF
stream_mode: updates
content: {'tools': {'messages': [ToolMessage(content="It's always sunny in SF!", name='get_weather', id='a83f09c

## 7. 常见模式
以下是展示流式传输常见用例的示例。
### 7.1 流式传输工具调用(Streaming Tools Calls)
可以同时流式传输两项内容：
- 工具调用生成过程中的部分 JSON 数据
- 经解析并待执行的完整工具调用指令

指定 `stream_mode="messages"` 时，将流式传输智能体中所有大语言模型调用生成的增量消息块。若要获取包含已解析工具调用的完整消息：

- 若这些消息在状态中被追踪（如 create_agent 的模型节点中），可通过 stream_mode=["messages", "updates"] 借助状态更新来获取完整消息（下文示例）。
- 若这些消息未在状态中被追踪，请使用自定义更新，或在流式传输循环中聚合消息块（详见下一节）。

In [8]:
from typing import Any

from langchain.agents import create_agent
from langchain.messages import AIMessage, AIMessageChunk, AnyMessage, ToolMessage
from langchain_ollama import ChatOllama

def get_weather(city: str) -> str:
    """Get weather for a given city."""

    return f"It's always sunny in {city}!"

model = ChatOllama(model="qwen3:0.6b")

agent = create_agent(model, tools=[get_weather])


def _render_message_chunk(token: AIMessageChunk) -> None:
    if token.text:
        print(token.text, end="|")
    if token.tool_call_chunks:
        print(token.tool_call_chunks)
    # N.B. all content is available through token.content_blocks


def _render_completed_message(message: AnyMessage) -> None:
    if isinstance(message, AIMessage) and message.tool_calls:
        print(f"Tool calls: {message.tool_calls}")
    if isinstance(message, ToolMessage):
        print(f"Tool response: {message.content_blocks}")


input_message = {"role": "user", "content": "What is the weather in Boston?"}
for stream_mode, data in agent.stream(
    {"messages": [input_message]},
    stream_mode=["messages", "updates"],
):
    if stream_mode == "messages":
        token, metadata = data
        if isinstance(token, AIMessageChunk):
            _render_message_chunk(token)
    if stream_mode == "updates":
        for source, update in data.items():
            if source in ("model", "tools"):  # `source` captures node name
                _render_completed_message(update["messages"][-1])

[{'name': 'get_weather', 'args': '{"city": "Boston"}', 'id': 'd76fd45e-0e34-4842-99c3-1cc9f7c951bd', 'index': None, 'type': 'tool_call_chunk'}]
Tool calls: [{'name': 'get_weather', 'args': {'city': 'Boston'}, 'id': 'd76fd45e-0e34-4842-99c3-1cc9f7c951bd', 'type': 'tool_call'}]
Tool response: [{'type': 'text', 'text': "It's always sunny in Boston!"}]
It|'s| always| sunny| in| Boston|!| 🌤|️|